In [1]:
# Assembling the Transformer: the code from the post, plus the two figures.
# pip install jax flax matplotlib

In [2]:
import jax
import jax.numpy as jnp
from flax import nnx


class Head(nnx.Module):
    """One self attention head scaled and masked."""

    def __init__(self, d_model: int, d_k: int, *, rngs: nnx.Rngs):
        self.d_k = d_k
        self.w_q = nnx.Linear(d_model, d_k, use_bias=False, rngs=rngs)
        self.w_k = nnx.Linear(d_model, d_k, use_bias=False, rngs=rngs)
        self.w_v = nnx.Linear(d_model, d_k, use_bias=False, rngs=rngs)

    def __call__(self, x, mask):
        queries, keys, values = self.w_q(x), self.w_k(x), self.w_v(x)   # (b, n, d_k)

        scores = queries @ keys.swapaxes(-1, -2) / jnp.sqrt(self.d_k)   # (b, n, n)
        scores = jnp.where(mask, scores, -jnp.inf)                      # before the softmax

        weights = jax.nn.softmax(scores, axis=-1)   # each row sums over its prefix
        return weights @ values                     # (b, n, d_k)


class MultiHeadAttention(nnx.Module):
    """Multi-head causal self-attention over a batch of sequences."""

    def __init__(self, d_model: int, n_heads: int, *, rngs: nnx.Rngs):
        assert d_model % n_heads == 0
        self.heads = nnx.List([
            Head(d_model, d_model // n_heads, rngs=rngs) for _ in range(n_heads)
        ])
        self.w_o = nnx.Linear(d_model, d_model, use_bias=False, rngs=rngs)

    def __call__(self, x):
        n = x.shape[-2]
        causal = jnp.tril(jnp.ones((n, n), dtype=bool))   # True on and below the diagonal

        outputs = [head(x, causal) for head in self.heads]   # each (b, n, d_k)
        return self.w_o(jnp.concatenate(outputs, axis=-1))   # (b, n, d_model)

In [3]:
class Block(nnx.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, *, rngs: nnx.Rngs):
        self.norm1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.attn = MultiHeadAttention(d_model, n_heads, rngs=rngs)
        self.norm2 = nnx.LayerNorm(d_model, rngs=rngs)
        self.ff_in = nnx.Linear(d_model, d_ff, rngs=rngs)
        self.ff_out = nnx.Linear(d_ff, d_model, rngs=rngs)

    def __call__(self, x):
        x = x + self.attn(self.norm1(x))                          # mix across positions
        return x + self.ff_out(jax.nn.relu(self.ff_in(self.norm2(x))))   # process each position

In [4]:
def positional_encoding(n: int, d_model: int):
    pos = jnp.arange(n)[:, None]                       # (n, 1)
    i = jnp.arange(0, d_model, 2)[None, :]             # (1, d_model/2)
    angle = pos / jnp.power(10000, i / d_model)
    pe = jnp.zeros((n, d_model))
    pe = pe.at[:, 0::2].set(jnp.sin(angle))            # even dimensions
    pe = pe.at[:, 1::2].set(jnp.cos(angle))            # odd dimensions
    return pe

In [5]:
class DecoderOnlyTransformer(nnx.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len, *, rngs):
        self.embed = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.pos = positional_encoding(max_len, d_model)
        self.blocks = nnx.List([
            Block(d_model, n_heads, d_ff, rngs=rngs) for _ in range(n_layers)
        ])
        self.norm = nnx.LayerNorm(d_model, rngs=rngs)
        self.unembed = nnx.Linear(d_model, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, token_ids):
        n = token_ids.shape[-1]
        x = self.embed(token_ids) + self.pos[:n]   # (b, n, d_model)
        for block in self.blocks:
            x = block(x)
        return self.unembed(self.norm(x))          # (b, n, vocab_size)

In [6]:
model = DecoderOnlyTransformer(
    vocab_size=1000, d_model=64, n_heads=4, d_ff=256,
    n_layers=2, max_len=128, rngs=nnx.Rngs(0),
)

tokens = jnp.zeros((1, 12), dtype=jnp.int32)   # one sequence of 12 token ids
logits = model(tokens)

n_params = sum(p.size for p in jax.tree.leaves(nnx.state(model, nnx.Param)))
print(logits.shape, n_params)   # (1, 12, 1000) 227584

(1, 12, 1000) 227584


In [7]:
# where the parameters sit in one block: attention vs feed-forward
block = model.blocks[0]
count = lambda m: sum(p.size for p in jax.tree.leaves(nnx.state(m, nnx.Param)))
print('attention', count(block.attn))
print('feed-forward', count(block.ff_in) + count(block.ff_out))

attention 16384
feed-forward 33088


In [8]:
# a masked attention weight matrix: every row is still a distribution
scores = jnp.ones((4, 4))
masked = jnp.where(jnp.tril(jnp.ones((4, 4), dtype=bool)), scores, -jnp.inf)
weights = jax.nn.softmax(masked, axis=-1)
print(weights)
print(weights.sum(axis=-1))

[[1.         0.         0.         0.        ]
 [0.5        0.5        0.         0.        ]
 [0.33333334 0.33333334 0.33333334 0.        ]
 [0.25       0.25       0.25       0.25      ]]
[1. 1. 1. 1.]


In [9]:
# ---- figures ----
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

TEXT = "#94a3b8"
GRID = "#334155"
LINE = "#e2e8f0"
AMBER = "#f59e0b"
CYAN = "#38bdf8"
RED = "#f87171"

plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "savefig.facecolor": "none",
    "savefig.transparent": True,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "axes.edgecolor": GRID,
    "xtick.color": TEXT,
    "ytick.color": TEXT,
    "grid.color": GRID,
    "font.size": 11,
})

OUT = "/mnt/jagah/naman-sopho.github.io/assets/img"
rng = np.random.default_rng(0)


def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

In [10]:
# figure: softmax saturation, assets/img/softmax-saturation.svg
# left: one query's attention weights over 8 keys, with and without 1/sqrt(d_k)
# right: how flat the softmax gets as d_k grows, measured by sum p(1-p)
d_k = 64
n_keys = 8
q = rng.normal(size=d_k)
K = rng.normal(size=(n_keys, d_k))
scores = K @ q

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

idx = np.arange(n_keys)
w_raw = softmax(scores)
w_scaled = softmax(scores / np.sqrt(d_k))
ax1.bar(idx - 0.2, w_raw, width=0.4, color=AMBER, label="unscaled $q \\cdot k$")
ax1.bar(idx + 0.2, w_scaled, width=0.4, color=CYAN, label="scaled by $1/\\sqrt{d_k}$")
ax1.set_xlabel("key position")
ax1.set_ylabel("attention weight")
ax1.set_xticks(idx)
ax1.set_ylim(0, 1.05)
ax1.legend(frameon=False, labelcolor=TEXT)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", alpha=0.3)
ax1.set_axisbelow(True)

dims = np.array([2, 4, 8, 16, 32, 64, 128, 256, 512])
trials = 200
raw_grad, scaled_grad = [], []
for d in dims:
    qq = rng.normal(size=(trials, d))
    KK = rng.normal(size=(trials, n_keys, d))
    s = np.einsum("tkd,td->tk", KK, qq)
    raw_grad.append(np.mean(np.sum(softmax(s) * (1 - softmax(s)), axis=-1)))
    p = softmax(s / np.sqrt(d))
    scaled_grad.append(np.mean(np.sum(p * (1 - p), axis=-1)))

ax2.plot(dims, raw_grad, "o-", color=AMBER, label="unscaled")
ax2.plot(dims, scaled_grad, "o-", color=CYAN, label="scaled by $1/\\sqrt{d_k}$")
ax2.set_xscale("log", base=2)
ax2.set_xlabel("$d_k$")
ax2.set_ylabel("softmax gradient scale\n$\\sum_i p_i(1 - p_i)$")
ax2.set_ylim(0, 1.05)
ax2.legend(frameon=False, labelcolor=TEXT)
ax2.spines[["top", "right"]].set_visible(False)
ax2.grid(alpha=0.3)
ax2.set_axisbelow(True)

fig.tight_layout()
fig.savefig(f"{OUT}/softmax-saturation.svg", bbox_inches="tight")
plt.close(fig)
print("raw_grad", [round(v, 3) for v in raw_grad])
print("scaled_grad", [round(v, 3) for v in scaled_grad])
print("w_raw", np.round(w_raw, 3), "max", w_raw.max().round(3))
print("w_scaled", np.round(w_scaled, 3), "max", w_scaled.max().round(3))

raw_grad [np.float64(0.737), np.float64(0.621), np.float64(0.496), np.float64(0.363), np.float64(0.249), np.float64(0.193), np.float64(0.138), np.float64(0.06), np.float64(0.07)]
scaled_grad [np.float64(0.795), np.float64(0.78), np.float64(0.777), np.float64(0.772), np.float64(0.775), np.float64(0.769), np.float64(0.766), np.float64(0.764), np.float64(0.77)]
w_raw [0.    0.    0.    0.    0.982 0.    0.018 0.   ] max 0.982
w_scaled [0.048 0.068 0.125 0.064 0.346 0.101 0.21  0.038] max 0.346


In [11]:
# figure: sinusoidal positional encoding, assets/img/positional-encoding.svg
# sinusoidal positional encoding heatmap
d_model, max_len = 128, 64
pos = np.arange(max_len)[:, None]
i = np.arange(0, d_model, 2)[None, :]
angle = pos / np.power(10000, i / d_model)
pe = np.zeros((max_len, d_model))
pe[:, 0::2] = np.sin(angle)
pe[:, 1::2] = np.cos(angle)

cmap = LinearSegmentedColormap.from_list("site", [CYAN, "#0f172a", AMBER])
fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.0), width_ratios=[1.25, 1])

# even dimensions only: the sine channels, ordered by increasing wavelength
im = axl.imshow(pe[:, 0::2], aspect="auto", cmap=cmap, vmin=-1, vmax=1,
                interpolation="nearest")
axl.set_xlabel("sine channel (even embedding dimensions)")
axl.set_ylabel("position in sequence")
cbar = fig.colorbar(im, ax=axl)
cbar.outline.set_edgecolor(GRID)
cbar.ax.tick_params(color=TEXT)
for spine in axl.spines.values():
    spine.set_edgecolor(GRID)

for dim, colour in zip([0, 8, 24, 48], [CYAN, AMBER, RED, LINE]):
    axr.plot(np.arange(max_len), pe[:, dim], color=colour, label=f"dimension {dim}")
axr.set_xlabel("position in sequence")
axr.set_ylabel("encoding value")
axr.set_ylim(-1.25, 1.6)
axr.legend(frameon=False, labelcolor=TEXT, ncol=2, loc="upper center")
axr.spines[["top", "right"]].set_visible(False)
axr.grid(alpha=0.3)
axr.set_axisbelow(True)

fig.tight_layout()
fig.savefig(f"{OUT}/positional-encoding.svg", bbox_inches="tight")
plt.close(fig)
print("pe range", pe.min().round(3), pe.max().round(3))

pe range -1.0 1.0


In [ ]:
# figure: three transformers, assets/img/three-transformers.png
# Panels 1 and 2 are public domain patent drawings from Wikimedia Commons,
# recoloured to light-on-transparent; panel 3 is drawn here in the same style.
import io
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrow, Rectangle
from PIL import Image

OUT = "/mnt/jagah/naman-sopho.github.io/assets/img"
INK = "#E2E8F0"
MUTED = "#94A3B8"
RULE = "#334155"
SERIF = ["Nimbus Roman", "Times New Roman", "Liberation Serif", "DejaVu Serif"]

SOURCES = {
    # W. Stanley Jr., Induction Coil, US 349,611 (1886). Patent drawing, public domain.
    "stanley": ("https://upload.wikimedia.org/wikipedia/commons/4/4c/StanleyTransformer2.png",
                (700, 185, 1490, 890)),
    # H. Obara, reconfigurable toy assembly, US 4,516,948 (filed 1984). Public domain.
    "optimus": ("https://upload.wikimedia.org/wikipedia/commons/e/e6/Optimus_Prime_patent.png",
                (322, 250, 605, 632)),
}


def fetch(key):
    url, crop = SOURCES[key]
    req = urllib.request.Request(url, headers={"User-Agent": "naman-sopho.github.io figure build"})
    with urllib.request.urlopen(req, timeout=60) as r:
        im = Image.open(io.BytesIO(r.read())).convert("L")
    return np.asarray(im.crop(crop), dtype=float) / 255.0


def to_square(gray, pad_value=1.0):
    """Pad a crop out to a square so every panel shares one aspect ratio."""
    h, w = gray.shape
    side = max(h, w)
    out = np.full((side, side), pad_value)
    y, x = (side - h) // 2, (side - w) // 2
    out[y:y + h, x:x + w] = gray
    return out


def as_light_lines(gray, gamma=1.5):
    """Dark-on-white scan -> light lines on transparency, in the site's ink colour."""
    alpha = np.clip(1.0 - gray, 0, 1) ** (1 / gamma)
    rgb = [int(INK[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    return np.dstack([np.ones_like(alpha) * c for c in rgb] + [alpha])


def box(ax, x, y, w, h, label, fontsize=9, lw=1.2):
    ax.add_patch(Rectangle((x, y), w, h, fill=False, ec=INK, lw=lw))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
            family=SERIF, fontsize=fontsize, color=INK, linespacing=1.3)


def arrow(ax, x0, y0, x1, y1, head=0.024):
    ax.add_patch(FancyArrow(x0, y0, x1 - x0, y1 - y0, width=0.0018, head_width=head,
                            head_length=head * 1.1, length_includes_head=True,
                            fc=INK, ec=INK, lw=0.5))


def leader(ax, num, tx, ty, x0, y0):
    """Patent-style numbered callout: italic numeral with a leader line to the part."""
    ax.plot([tx, x0], [ty, y0], color=INK, lw=0.8)
    ax.text(tx, ty, num, ha="center", va="center", family=SERIF, style="italic",
            fontsize=11, color=INK)


def draw_attention_figure(ax):
    """The stack built in this post, drawn as a patent figure."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    ax.text(0.90, 0.945, "Fig. 1.", family=SERIF, style="italic", fontsize=13,
            color=INK, ha="right")

    # input tokens
    for i in range(5):
        ax.add_patch(Rectangle((0.335 + i * 0.075, 0.035), 0.052, 0.040,
                               fill=False, ec=INK, lw=1.0))
    leader(ax, "10", 0.22, 0.055, 0.330, 0.055)

    arrow(ax, 0.5, 0.080, 0.5, 0.125)
    box(ax, 0.30, 0.125, 0.40, 0.070, "TOKEN EMBEDDING\n+ POSITION", fontsize=8)
    leader(ax, "12", 0.22, 0.160, 0.295, 0.160)

    # the repeated block
    ax.add_patch(Rectangle((0.185, 0.245), 0.63, 0.415, fill=False, ec=INK, lw=1.3,
                           linestyle=(0, (5, 3))))
    ax.text(0.50, 0.600, "BLOCK, REPEATED N TIMES", ha="center", va="center",
            family=SERIF, fontsize=8, color=MUTED)
    leader(ax, "14", 0.115, 0.450, 0.183, 0.450)

    arrow(ax, 0.5, 0.195, 0.5, 0.285)
    box(ax, 0.295, 0.285, 0.41, 0.095, "MASKED\nSELF-ATTENTION", fontsize=8.5)
    leader(ax, "16", 0.885, 0.332, 0.710, 0.332)

    arrow(ax, 0.5, 0.380, 0.5, 0.440)
    box(ax, 0.295, 0.440, 0.41, 0.095, "FEED-FORWARD\nNETWORK", fontsize=8.5)
    leader(ax, "18", 0.885, 0.487, 0.710, 0.487)

    # residual bypasses around each sublayer
    for y0, y1 in [(0.265, 0.410), (0.415, 0.560)]:
        ax.plot([0.245, 0.245], [y0, y1], color=INK, lw=1.0)
        ax.plot([0.245, 0.292], [y0, y0], color=INK, lw=1.0)
        arrow(ax, 0.245, y1, 0.290, y1, head=0.018)
    leader(ax, "20", 0.135, 0.560, 0.243, 0.520)

    arrow(ax, 0.5, 0.535, 0.5, 0.715)
    box(ax, 0.30, 0.715, 0.40, 0.070, "LINEAR TO VOCABULARY", fontsize=8)
    leader(ax, "22", 0.885, 0.750, 0.705, 0.750)

    arrow(ax, 0.5, 0.785, 0.5, 0.855)
    ax.text(0.50, 0.870, "NEXT TOKEN", ha="center", va="bottom", family=SERIF,
            fontsize=9, color=INK)


fig = plt.figure(figsize=(13.6, 6.0), facecolor="none")

# three square panels on one baseline, so the headers and footers line up
side_w = 0.275
side_h = side_w * 13.6 / 6.0
y0 = 0.155
centres = [0.175, 0.5, 0.825]
axes = [fig.add_axes([c - side_w / 2, y0, side_w, side_h]) for c in centres]
for ax in axes:
    ax.set_facecolor("none")
    ax.axis("off")

panels = [
    ("W. STANLEY, JR.", "INDUCTION COIL", "No. 349,611", "1886"),
    ("H. OBARA", "RECONFIGURABLE TOY ASSEMBLY", "No. 4,516,948", "filed 1984"),
    ("VASWANI ET AL.", "ATTENTION IS ALL YOU NEED", "arXiv:1706.03762", "2017"),
]

for ax, key in zip(axes[:2], ["stanley", "optimus"]):
    ax.imshow(as_light_lines(to_square(fetch(key))), interpolation="lanczos")
draw_attention_figure(axes[2])

top = y0 + side_h
for c, (name, subject, number, year) in zip(centres, panels):
    fig.text(c, top + 0.075, name, ha="center", va="bottom", family=SERIF,
             fontsize=16, color=INK)
    fig.text(c, top + 0.025, subject, ha="center", va="bottom", family=SERIF,
             fontsize=11, color=MUTED)
    fig.text(c - side_w / 2, y0 - 0.045, number, ha="left", va="top", family=SERIF,
             fontsize=11, color=MUTED)
    fig.text(c + side_w / 2, y0 - 0.045, year, ha="right", va="top", family=SERIF,
             fontsize=11, color=MUTED)

for x in (0.3375, 0.6625):
    fig.add_artist(plt.Line2D([x, x], [y0 - 0.02, top + 0.10], color=RULE, lw=1.0))

fig.savefig(f"{OUT}/three-transformers.png", dpi=150, transparent=True,
            bbox_inches="tight", pad_inches=0.3)
plt.close(fig)
print("wrote", f"{OUT}/three-transformers.png")